In [ ]:
from pathlib import Path
from typing import Literal

import h5py
import requests
import pandas as pd
from tqdm import tqdm

GDC_BASE = "https://api.gdc.cancer.gov"
PATCH_DIR = Path("../data/WSIs/patch-embeddings")
OUT_DIR = Path("../embeddings/WSIs")
COHORT_T = Literal["TCGA-LUAD", "TCGA-LUSC", "CPTAC-LUAD", "CPTAC-LSCC"]

In [ ]:
def make_filter(cohort: COHORT_T):
    match cohort:
        case "TCGA-LUAD":
            project = "TCGA-LUAD"
            disease_type = "adenomas and adenocarcinomas"
        case "TCGA-LUSC":
            project = "TCGA-LUSC"
            disease_type = "squamous cell neoplasms"
        case "CPTAC-LUAD":
            project = "CPTAC-3"
            disease_type = "adenomas and adenocarcinomas"
        case "CPTAC-LSCC":
            project = "CPTAC-3"
            disease_type = "squamous cell neoplasms"
        case _:
            raise ValueError(f"Unknown project/disease_type values for {cohort}")

    # can't select on slide modality for CPTAC in GDC (CPTAC images are from IDC/TCIA)
    # but the GDC does index which samples had slides so just retrieve case-level metadata
    # and we'll filter to tumor/solid tissue/primary samples after, then select their slides
    return {
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "cases.project.project_id", "value": [project]}},
            {"op": "in", "content": {"field": "cases.disease_type", "value": [disease_type]}},
            {"op": "in", "content": {"field": "cases.primary_site", "value": ["bronchus and lung"]}},
        ],
    }

In [ ]:
def get_tumor_meta(cohort: COHORT_T):
    temp = requests.post(
        f"{GDC_BASE}/cases",
        json={
            "filters": make_filter(cohort),
            "fields": ",".join([
                "submitter_id",
                "samples.tissue_type", "samples.specimen_type", "samples.tumor_descriptor",
                "samples.portions.slides.slide_id", "samples.portions.slides.submitter_id",
            ]),
            "size": "100000",
        }
    )
    temp.raise_for_status()
    hits = temp.json()["data"]["hits"]

    # get only slides belonging to primary tumor
    tumor_meta = []
    for hit in hits:
        for sample in hit["samples"]:
            if (
                sample["tumor_descriptor"] != "Primary"
                or sample["specimen_type"] != "Solid Tissue"
                or sample["tissue_type"] != "Tumor"
                or "portions" not in sample
            ):
                continue
            for portion in sample["portions"]:
                for slide in portion["slides"]:
                    tumor_meta.append({
                        "case_id": hit["submitter_id"],
                        "slide_id": slide["submitter_id"],
                    })
    tumor_df = pd.DataFrame(tumor_meta)
    tumor_df = tumor_df.drop_duplicates().reset_index(drop=True) # drop exact dupes based on case_id/slide_id
    return tumor_df

In [ ]:
def get_slide_meta(cohort: COHORT_T):
    slide_meta = []
    for fpath in (PATCH_DIR / cohort).iterdir():
        # CPTAC style: C3L-00001-21.h5
        # TCGA style: TCGA-05-4244-01Z-00-DX1.d4ff32cd-38cf-40ea-8213-45c2b100ac01.h5
        slide_id = fpath.name.split(".")[0]
        slide_meta.append({
            "slide_id": slide_id,
            "fpath": fpath,
        })
    slide_df = pd.DataFrame(slide_meta)
    return slide_df

In [ ]:
def make_cohort_slide_embs(cohort: COHORT_T):
    tumor_df = get_tumor_meta(cohort)
    slide_df = get_slide_meta(cohort)
    assert tumor_df["slide_id"].is_unique
    assert slide_df["slide_id"].is_unique
    df = tumor_df.merge(slide_df)
    df = df.sort_values(["case_id", "slide_id"]).reset_index(drop=True)

    OUT_DIR.mkdir(parents=True, exist_ok=True)

    with h5py.File(OUT_DIR / f"{cohort}.h5", "w") as h5_out:
        _fpaths = tqdm(df["fpath"], desc=f"Making slide embs for {cohort}")
        for fpath, case_id in zip(_fpaths, df["case_id"]):
            with h5py.File(fpath, "r") as h5_in:
                emb = h5_in["features"][:].mean(axis=(0, 1)) # type: ignore
            fname = fpath.name.replace(".h5", "")
            if case_id not in h5_out:
                h5_out.create_group(case_id)
            h5_out[case_id].create_dataset(fname, data=emb)

In [ ]:
make_cohort_slide_embs("TCGA-LUAD")
make_cohort_slide_embs("TCGA-LUSC")
make_cohort_slide_embs("CPTAC-LUAD")
make_cohort_slide_embs("CPTAC-LSCC")